# GenerativeAURORA — Step 5: Train Diffusion Model

## Purpose

This notebook is a **faithful notebook conversion of `step5_train.py`**. The original training logic is preserved rather than redesigned.

Step 5 trains the conditional diffusion model defined in **Step 4** using the sequence datasets produced by **Step 3**.

### Main features from the original script
- Runs on CUDA automatically when available.
- Uses a weighted sampler to compensate for sky-regime imbalance.
- Saves a checkpoint after every epoch.
- Automatically resumes from the latest checkpoint when one exists.
- Tracks validation loss every epoch.
- Saves the best model separately as `best_model.pt`.

### Important conversion rule
The model architecture is **not copied into this notebook**. As in the original `.py` file, `GenerativeAURORA` is imported from `step4_build_model.py`. This keeps Step 4 responsible for the architecture and Step 5 responsible for training.

> **Note for Colab:** The original script contains Windows paths such as `C:/Users/user/Paper_10/...`. They are intentionally preserved below so that the code matches the supplied `.py` exactly. Before executing in Colab, those environment-specific paths need to point to the corresponding Colab/Google Drive locations.

## 1. Original script header and imports

The following cell contains the original imports used by the training script. No additional model architecture is introduced here.

In [ ]:
# GenerativeAURORA - Step 5: Train Diffusion Model
#
# Features:
#   - Runs on RTX 4080 Super (CUDA) automatically
#   - Weighted sampler to handle regime imbalance (Clear 93%, Cloudy 6%, Overcast 1%)
#   - Checkpoint saved after every epoch to Dataset/checkpoints/
#   - Resumes automatically from last checkpoint if power cuts
#   - Validation loss tracked every epoch
#   - Best model saved separately as best_model.pt
#
# Run:
#   python step5_train.py

import os
import json
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Paths

These are the same paths used by the original script. `DATASET_DIR` contains the sequence datasets and training outputs, while `CODING_DIR` contains `step4_build_model.py`. The checkpoint directory is created automatically.

In [ ]:
# -----------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------
DATASET_DIR  = "/content/drive/MyDrive/Colab Notebooks/Dataset"
CODING_DIR   = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"
CKPT_DIR     = os.path.join(DATASET_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)


## 3. Training hyperparameters

These values are taken directly from the original script. The model dimensions are required to match the architecture defined in Step 4.

In [ ]:
# -----------------------------------------------------------------------
# TRAINING HYPERPARAMETERS
# -----------------------------------------------------------------------
BATCH_SIZE    = 128
EPOCHS        = 200
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
GRAD_CLIP     = 1.0
T_DIFFUSION   = 200

# Model dimensions (must match step4_build_model.py)
HORIZON     = 16
LOOKBACK    = 16
N_FEATURES  = 10
N_REGIMES   = 3
N_SITES     = 5
EMBED_DIM   = 64
HIDDEN_DIM  = 128


## 4. Import the model from Step 4

The original training script imports `GenerativeAURORA` rather than redefining the architecture. This preserves the separation between **model construction (Step 4)** and **model training (Step 5)**.

In [ ]:
# -----------------------------------------------------------------------
# IMPORT MODEL FROM STEP 4
# -----------------------------------------------------------------------
import os
import json
import sys

# Path to the target notebook on Google Drive
notebook_path = "/content/drive/MyDrive/Colab Notebooks/GenerativeAURORA_step4_build_model_Colab.ipynb"

if not os.path.exists(notebook_path):
    raise FileNotFoundError(f"Could not find notebook at: {notebook_path}. Please check your Google Drive mount or path.")

# Read and execute Python code blocks from the step 4 notebook directly
with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Execute code cells sequentially in the global namespace
for cell in nb_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        code_text = ''.join(cell.get('source', []))
        # Avoid executing installations or imports of itself recursively
        if "import_ipynb" not in code_text and "GenerativeAURORA_step4_build_model_Colab" not in code_text:
            exec(code_text, globals())

PyTorch version: 2.11.0+cpu
CUDA available  : False
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HORIZON     : 16
LOOKBACK    : 16
N_FEATURES  : 10
N_REGIMES   : 3
N_SITES     : 5
T_DIFFUSION : 200
EMBED_DIM   : 64
HIDDEN_DIM  : 128
Betas shape     : (200,)
Alphas shape    : (200,)
Alpha-bar shape : (200,)
First beta      : 9.999999747378752e-05
Last beta       : 0.019999999552965164
Device              : cpu
Total parameters    : 5,095,329
Trainable parameters: 5,095,329
x0 shape       : (4, 16)
cond_x shape   : (4, 16, 10)
regime shape   : (4,)
site shape     : (4,)
Loss value     : 0.93449
Loss dimensions: () (scalar expected)

Forward-pass shape check: PASSED
Generated scenarios shape: (4, 16)
Expected shape           : (4, 16)
Scenario-generation shape check: PASSED


## 5. Dataset class

`SolarSequenceDataset` loads the `.npz` files created in Step 3. Each sample contains:

- `X`: historical conditioning sequence
- `Y`: target GHI sequence
- `regime`: sky-regime index
- `site`: site index

The class converts these arrays to PyTorch tensors and returns one complete training sample.

In [ ]:
# -----------------------------------------------------------------------
# DATASET CLASS
# -----------------------------------------------------------------------

class SolarSequenceDataset(Dataset):
    def __init__(self, npz_path):
        data       = np.load(npz_path)
        self.X      = torch.tensor(data["X"],      dtype=torch.float32)
        self.Y      = torch.tensor(data["Y"],      dtype=torch.float32)
        self.regime = torch.tensor(data["regime"], dtype=torch.long)
        self.site   = torch.tensor(data["site"],   dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.regime[idx], self.site[idx]


## 6. Weighted sampler

The original script uses inverse-frequency sampling because the sky-regime distribution is highly imbalanced. Rare regimes therefore receive larger sampling weights during training.

In [ ]:
# -----------------------------------------------------------------------
# WEIGHTED SAMPLER (handles Clear 93% / Cloudy 6% / Overcast 1%)
# -----------------------------------------------------------------------

def make_weighted_sampler(dataset):
    regime_labels = dataset.regime.numpy()
    unique, counts = np.unique(regime_labels, return_counts=True)
    freq = dict(zip(unique.tolist(), counts.tolist()))

    # Weight = inverse frequency so rare regimes are sampled more
    weights = np.array([1.0 / freq[int(r)] for r in regime_labels], dtype=np.float32)
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(weights),
        replacement=True,
    )

    regime_names = {0: "Clear", 1: "Cloudy", 2: "Overcast"}
    print("  Regime weights (inverse frequency):")
    for k, v in freq.items():
        w = 1.0 / v
        print("    " + regime_names.get(k, str(k)).ljust(10)
              + " count: " + str(v).rjust(5)
              + "  weight: " + str(round(w, 6)))
    return sampler


## 7. Checkpoint utilities

The training script saves the model state, optimizer state, epoch, training loss, and validation loss. This allows training to resume after an interruption.

In [ ]:
# -----------------------------------------------------------------------
# CHECKPOINT UTILITIES
# -----------------------------------------------------------------------

def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, path):
    torch.save({
        "epoch":      epoch,
        "model":      model.state_dict(),
        "optimizer":  optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss":   val_loss,
    }, path)


def load_checkpoint(model, optimizer, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    best_val    = ckpt["val_loss"]
    print("Resumed from epoch " + str(ckpt["epoch"])
          + "  val_loss: " + str(round(best_val, 6)))
    return start_epoch, best_val


def find_latest_checkpoint(ckpt_dir):
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("epoch_") and f.endswith(".pt")]
    if not files:
        return None
    epochs = [int(f.replace("epoch_", "").replace(".pt", "")) for f in files]
    latest = max(epochs)
    return os.path.join(ckpt_dir, "epoch_" + str(latest) + ".pt")


## 8. Training and validation functions

`train_one_epoch()` performs the optimization step for each batch. `validate()` evaluates the model without gradient updates. Both functions call the same `GenerativeAURORA` model with `(Y, X, regime, site)`.

In [ ]:
# -----------------------------------------------------------------------
# TRAINING LOOP
# -----------------------------------------------------------------------

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches  = 0

    for X, Y, regime, site in loader:
        X      = X.to(device)
        Y      = Y.to(device)
        regime = regime.to(device)
        site   = site.to(device)

        optimizer.zero_grad()
        loss = model(Y, X, regime, site)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    with torch.no_grad():
        for X, Y, regime, site in loader:
            X      = X.to(device)
            Y      = Y.to(device)
            regime = regime.to(device)
            site   = site.to(device)

            loss = model(Y, X, regime, site)
            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches


## 9. Main training workflow

The `main()` function below is preserved from the original script. It:

1. selects CUDA when available;
2. loads the training and validation sequence datasets;
3. creates the weighted training sampler;
4. creates the DataLoaders;
5. instantiates `GenerativeAURORA` from Step 4;
6. creates AdamW and cosine-annealing learning-rate scheduling;
7. resumes from the latest checkpoint if available;
8. trains for the configured number of epochs;
9. saves epoch checkpoints and the best model; and
10. records the training history in `training_log.csv`.

In [ ]:
# -----------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------

def main():
    print("GenerativeAURORA - Step 5: Training")
    print("")

    # Device
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        print("Device : " + gpu_name + "  (" + str(vram) + " GB VRAM)")
    else:
        device = torch.device("cpu")
        print("Device : CPU  (CUDA not available -- install PyTorch with CUDA)")
        print("         pip install torch --index-url https://download.pytorch.org/whl/cu121")
    print("")

    # Datasets
    train_path = os.path.join(DATASET_DIR, "sequences_train.npz")
    val_path   = os.path.join(DATASET_DIR, "sequences_val.npz")

    print("Loading datasets...")
    train_ds = SolarSequenceDataset(train_path)
    val_ds   = SolarSequenceDataset(val_path)
    print("  Train: " + str(len(train_ds)) + " sequences")
    print("  Val  : " + str(len(val_ds))   + " sequences")
    print("")

    # Weighted sampler for train only
    print("Building weighted sampler for regime imbalance...")
    sampler = make_weighted_sampler(train_ds)
    print("")

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    # Model
    model = GenerativeAURORA(T=T_DIFFUSION).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print("Model parameters: " + str(total_params))
    print("")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6
    )

    # Resume from checkpoint if available
    start_epoch = 0
    best_val    = float("inf")
    latest_ckpt = find_latest_checkpoint(CKPT_DIR)

    if latest_ckpt is not None:
        print("Found checkpoint: " + latest_ckpt)
        start_epoch, best_val = load_checkpoint(model, optimizer, latest_ckpt, device)
        print("")
    else:
        print("No checkpoint found. Starting from scratch.")
        print("")

    # Training log
    log_path = os.path.join(DATASET_DIR, "training_log.csv")
    if start_epoch == 0:
        with open(log_path, "w", encoding="utf-8") as f:
            f.write("epoch,train_loss,val_loss,lr,elapsed_sec\n")

    print("Starting training...")
    print("Epochs       : " + str(EPOCHS))
    print("Batch size   : " + str(BATCH_SIZE))
    print("LR           : " + str(LR))
    print("Grad clip    : " + str(GRAD_CLIP))
    print("T diffusion  : " + str(T_DIFFUSION))
    print("Checkpoints  : " + CKPT_DIR)
    print("")
    print("-" * 65)

    t0_total = time.time()

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss   = validate(model, val_loader, device)
        scheduler.step()

        elapsed = round(time.time() - t0, 1)
        current_lr = round(scheduler.get_last_lr()[0], 8)

        # Print progress
        is_best = val_loss < best_val
        marker  = "  <-- best" if is_best else ""
        print("Epoch " + str(epoch + 1).rjust(3) + "/" + str(EPOCHS)
              + "  train: " + str(round(train_loss, 6)).ljust(10)
              + "  val: "   + str(round(val_loss,   6)).ljust(10)
              + "  lr: "    + str(current_lr).ljust(12)
              + "  " + str(elapsed) + "s"
              + marker)

        # Save checkpoint every epoch (enables resume after power cut)
        ckpt_path = os.path.join(CKPT_DIR, "epoch_" + str(epoch + 1) + ".pt")
        save_checkpoint(model, optimizer, epoch + 1, train_loss, val_loss, ckpt_path)

        # Keep only last 3 checkpoints to save disk space
        all_ckpts = sorted(
            [f for f in os.listdir(CKPT_DIR) if f.startswith("epoch_") and f.endswith(".pt")],
            key=lambda x: int(x.replace("epoch_", "").replace(".pt", ""))
        )
        for old in all_ckpts[:-3]:
            os.remove(os.path.join(CKPT_DIR, old))

        # Save best model separately
        if is_best:
            best_val = val_loss
            best_path = os.path.join(DATASET_DIR, "best_model.pt")
            torch.save(model.state_dict(), best_path)

        # Append to training log
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(str(epoch + 1) + ","
                    + str(round(train_loss, 6)) + ","
                    + str(round(val_loss, 6)) + ","
                    + str(current_lr) + ","
                    + str(elapsed) + "\n")

    total_time = round((time.time() - t0_total) / 60, 1)
    print("-" * 65)
    print("")
    print("Training complete.")
    print("Total time    : " + str(total_time) + " minutes")
    print("Best val loss : " + str(round(best_val, 6)))
    print("Best model    : " + os.path.join(DATASET_DIR, "best_model.pt"))
    print("Training log  : " + log_path)
    print("")
    print("Next: run step6_evaluate.py to generate scenarios and compute metrics.")


if __name__ == "__main__":
    main()


GenerativeAURORA - Step 5: Training

Device : CPU  (CUDA not available -- install PyTorch with CUDA)
         pip install torch --index-url https://download.pytorch.org/whl/cu121

Loading datasets...
  Train: 7305 sequences
  Val  : 905 sequences

Building weighted sampler for regime imbalance...
  Regime weights (inverse frequency):
    Clear      count:  6782  weight: 0.000147
    Cloudy     count:   501  weight: 0.001996
    Overcast   count:    22  weight: 0.045455

Model parameters: 5095329

Found checkpoint: /content/drive/MyDrive/Colab Notebooks/Dataset/checkpoints/epoch_200.pt
Resumed from epoch 200  val_loss: 0.025785

Starting training...
Epochs       : 200
Batch size   : 128
LR           : 0.0001
Grad clip    : 1.0
T diffusion  : 200
Checkpoints  : /content/drive/MyDrive/Colab Notebooks/Dataset/checkpoints

-----------------------------------------------------------------
-----------------------------------------------------------------

Training complete.
Total time    : 0.

## 10. Notebook execution note

In the original `.py` file, execution starts through:

```python
if __name__ == "__main__":
    main()
```

The same block is retained above so the notebook remains faithful to the source. In a notebook, you can also execute the final cell directly to start the training run.

### Expected outputs
- `Dataset/checkpoints/epoch_*.pt` — rolling training checkpoints
- `Dataset/best_model.pt` — best model according to validation loss
- `Dataset/training_log.csv` — epoch-by-epoch training history

### Next step
After training completes, proceed to **Step 6 — Evaluation and Scenario Generation**.